In [ ]:
import zipfile

with zipfile.ZipFile("chest xray data.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

In [ ]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_path = "dataset/Data/train"

datagen = ImageDataGenerator(rescale=1./255)

train_data = datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=8,
    class_mode='binary'
)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
import numpy as np

mobilenet = MobileNetV2(weights='imagenet', include_top=False)

x, y = next(train_data)
features = mobilenet.predict(x)

print("Image features shape:", features.shape)

In [ ]:
import pandas as pd

df = pd.read_csv("cvd_publication_ready_dataset.csv")

df["target"] = (df["target"] >= 0.5).astype(int)

X = df.drop("target", axis=1)
y = df["target"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
import numpy as np

def quantum_feature_map(X):
    return np.concatenate([np.sin(X), np.cos(X), X**2], axis=1)

X_train_q = quantum_feature_map(X_train)
X_test_q = quantum_feature_map(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

model.fit(X_train_q, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test_q)

accuracy = accuracy_score(y_test, y_pred)
print("Final Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

y_prob = model.predict_proba(X_test_q)[:,1]

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label="AUC = %.2f" % roc_auc)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.show()

In [ ]:
import shap

# create explainer
explainer = shap.KernelExplainer(model.predict_proba, X_train_q[:10])

# get shap values
shap_values = explainer.shap_values(X_test_q[:5])

# FIX: handle both cases safely
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[1], X_test_q[:5])
else:
    shap.summary_plot(shap_values, X_test_q[:5])

In [ ]:
!pip install lime

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

explainer = LimeTabularExplainer(
    X_train_q,
    feature_names=[f"f{i}" for i in range(X_train_q.shape[1])],
    class_names=["No Disease", "Disease"],
    mode='classification'
)

exp = explainer.explain_instance(
    X_test_q[0],
    model.predict_proba,
    num_features=10
)

exp.show_in_notebook()

In [ ]:
from tensorflow.keras.applications import ResNet50

resnet = ResNet50(weights='imagenet', include_top=False)

x, _ = next(train_data)
res_features = resnet.predict(x)

print(res_features.shape)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt

importances = model.feature_importances_

plt.plot(importances[:20])
plt.title("Top Feature Importance")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Compute correlation matrix
corr = df.corr()

# Plot heatmap
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm')

plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_pred = model.predict(X_test_q)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

df['target'].value_counts().plot(kind='bar')
plt.title("Target Class Distribution")
plt.xlabel("CVD (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.hist(df['age'], bins=20)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

X = df.drop('target', axis=1)
y = df['target']

model = RandomForestClassifier()
model.fit(X, y)

importances = model.feature_importances_

feat_imp = pd.Series(importances, index=X.columns)
feat_imp.sort_values().plot(kind='barh')

plt.title("Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier()
}

accuracies = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    accuracies.append(acc)

plt.bar(models.keys(), accuracies)
plt.title("Model Accuracy Comparison")
plt.xlabel("Models")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
print(df.columns)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.crosstab(df['sex'], df['target']).plot(kind='bar')

plt.title("Gender vs CVD")
plt.xlabel("Gender (0 = Female, 1 = Male)")
plt.ylabel("Count")
plt.legend(title="CVD")
plt.show()

In [ ]:
print(df.columns)

In [ ]:
import seaborn as sns

sns.boxplot(x='target', y='trestbps', data=df)

plt.title("Resting Blood Pressure vs CVD")
plt.xlabel("CVD")
plt.ylabel("trestbps")
plt.show()

In [ ]:
sns.boxplot(x='target', y='chol', data=df)

plt.title("Cholesterol vs CVD")
plt.xlabel("CVD")
plt.ylabel("Cholesterol")
plt.show()

In [ ]:
sns.boxplot(x='target', y='bmi', data=df)

plt.title("BMI vs CVD")
plt.xlabel("CVD")
plt.ylabel("BMI")
plt.show()

In [ ]:
sns.boxplot(x='target', y='stress_level', data=df)

plt.title("Stress Level vs CVD")
plt.xlabel("CVD")
plt.ylabel("Stress Level")
plt.show()

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred, output_dict=True)

table1 = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (Class 1)", "Recall (Class 1)", "F1-Score (Class 1)"],
    "Value": [
        accuracy,
        report['1']['precision'],
        report['1']['recall'],
        report['1']['f1-score']
    ]
})

print(table1)

In [ ]:
from sklearn.metrics import accuracy_score

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results.append([name, acc])

table2 = pd.DataFrame(results, columns=["Model", "Accuracy"])

print(table2)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

comparison = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    comparison.append([
        name,
        accuracy_score(y_test, preds),
        precision_score(y_test, preds),
        recall_score(y_test, preds),
        f1_score(y_test, preds)
    ])

table3 = pd.DataFrame(comparison,
                      columns=["Model", "Accuracy", "Precision", "Recall", "F1-score"])

print(table3)

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
import os
os.listdir()

In [ ]:
import os
os.listdir('dataset')

In [ ]:
os.listdir('dataset/Data')

In [ ]:
test_path = 'dataset/Data/test'

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

test_datagen = ImageDataGenerator(rescale=1./255)

test_gen = test_datagen.flow_from_directory(
    'dataset/Data/test',   # 🔥 FIXED PATH (change if needed)
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

In [ ]:
test_gen = test_datagen.flow_from_directory(
    'dataset/test',
    target_size=(224,224),
    batch_size=16,   # 🔥 reduce from 32 → 16
    class_mode='binary',
    shuffle=False
)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

test_datagen = ImageDataGenerator(rescale=1./255)

test_gen = test_datagen.flow_from_directory(
    'dataset/Data/test',   # 🔥 change if needed
    target_size=(224,224),
    batch_size=16,
    class_mode='binary',
    shuffle=False
)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

test_datagen = ImageDataGenerator(rescale=1./255)

test_gen = test_datagen.flow_from_directory(
    'dataset/Data/test',   # 🔥 change if needed
    target_size=(224,224),
    batch_size=16,
    class_mode='binary',
    shuffle=False
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# MobileNet
acc_mob = accuracy_score(y_test_images, y_pred_mobilenet)
prec_mob = precision_score(y_test_images, y_pred_mobilenet)
rec_mob = recall_score(y_test_images, y_pred_mobilenet)
f1_mob = f1_score(y_test_images, y_pred_mobilenet)

# ResNet
acc_res = accuracy_score(y_test_images, y_pred_resnet)
prec_res = precision_score(y_test_images, y_pred_resnet)
rec_res = recall_score(y_test_images, y_pred_resnet)
f1_res = f1_score(y_test_images, y_pred_resnet)

print("📊 MobileNet:", acc_mob, prec_mob, rec_mob, f1_mob)
print("📊 ResNet:", acc_res, prec_res, rec_res, f1_res)

In [ ]:
# MobileNet predictions
y_pred_mobilenet = model.predict(test_gen, verbose=1)
y_pred_mobilenet = (y_pred_mobilenet > 0.5).astype(int)

# ResNet predictions
y_pred_resnet = resnet_model.predict(test_gen, verbose=1)
y_pred_resnet = (y_pred_resnet > 0.5).astype(int)

# True labels (THIS LINE WAS MISSING ❗)
y_test_images = test_gen.classes

In [ ]:
# MobileNet predictions (FIXED)
y_pred_mobilenet = mobilenet_model.predict(test_gen)
y_pred_mobilenet = (y_pred_mobilenet > 0.5).astype(int)

# ResNet predictions
y_pred_resnet = resnet_model.predict(test_gen)
y_pred_resnet = (y_pred_resnet > 0.5).astype(int)

# True labels
y_test_images = test_gen.classes

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

mobilenet_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid')
])

mobilenet_model.compile(optimizer='adam',
                        loss='binary_crossentropy',
                        metrics=['accuracy'])

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2   # 80% train, 20% validation
)

# Train generator
train_gen = datagen.flow_from_directory(
    'dataset/Data/train',   # 🔥 adjust if needed
    target_size=(224,224),
    batch_size=16,
    class_mode='binary',
    subset='training'
)

# Validation generator
val_gen = datagen.flow_from_directory(
    'dataset/Data/train',
    target_size=(224,224),
    batch_size=16,
    class_mode='binary',
    subset='validation'
)

In [ ]:
mobilenet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5
)

In [ ]:
# MobileNet predictions
y_pred_mobilenet = mobilenet_model.predict(test_gen)
y_pred_mobilenet = (y_pred_mobilenet > 0.5).astype(int)

# True labels
y_test_images = test_gen.classes

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc_mob = accuracy_score(y_test_images, y_pred_mobilenet)
prec_mob = precision_score(y_test_images, y_pred_mobilenet)
rec_mob = recall_score(y_test_images, y_pred_mobilenet)
f1_mob = f1_score(y_test_images, y_pred_mobilenet)

print("📊 MobileNet Metrics")
print("Accuracy:", acc_mob)
print("Precision:", prec_mob)
print("Recall:", rec_mob)
print("F1-score:", f1_mob)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm_mob = confusion_matrix(y_test_images, y_pred_mobilenet)

plt.figure(figsize=(5,4))
sns.heatmap(cm_mob, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Healthy','CVD'],
            yticklabels=['Healthy','CVD'])
plt.title("MobileNet Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    'dataset/Data/train',   # 🔥 check path
    target_size=(224,224),
    batch_size=16,
    class_mode='binary',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    'dataset/Data/train',
    target_size=(224,224),
    batch_size=16,
    class_mode='binary',
    subset='validation'
)

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False   # 🔥 IMPORTANT

resnet_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid')
])

resnet_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
resnet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_gen = test_datagen.flow_from_directory(
    'dataset/Data/test',   # 🔥 same structure
    target_size=(224,224),
    batch_size=16,
    class_mode='binary',
    shuffle=False
)

In [ ]:
y_pred_resnet = resnet_model.predict(test_gen)
y_pred_resnet = (y_pred_resnet > 0.5).astype(int)

y_test_images = test_gen.classes